# 로컬 실험 노트북

이 노트북은 내 컴퓨터에서 실험을 돌리고 결과를 바로 확인하기 위한 템플릿입니다.

처음 보는 사람은 위에서부터 순서대로 실행하면 됩니다. 각 단계는 “무엇을 확인하는지”와 “어디가 이상하면 무엇을 의심해야 하는지”를 함께 적어두었습니다.

## 1. Bootstrap

노트북이 프로젝트 폴더를 제대로 바라보고 있는지 확인합니다.

정상이라면 `PROJECT_ROOT`가 이 저장소 경로로 출력됩니다. 여기서 에러가 나면 노트북을 프로젝트 루트가 아닌 다른 폴더에서 실행한 것입니다.

In [ ]:
from pathlib import Path
import json
import platform
import subprocess
import sys

import pandas as pd
import matplotlib.pyplot as plt

# 그래프 스타일을 조금 보기 좋게 맞춥니다.
plt.style.use("seaborn-v0_8-whitegrid")

# 노트북은 프로젝트 루트에서 실행하는 것을 기준으로 합니다.
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "configs").exists():
    raise RuntimeError("노트북을 프로젝트 루트에서 실행해주세요.")

# src/ 모듈을 노트북에서 import할 수 있게 프로젝트 루트를 Python 경로에 추가합니다.
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("PROJECT_ROOT:", PROJECT_ROOT)
print("Python:", platform.python_version())

## 2. 실험 옵션을 어떻게 바꾸나

새 실험을 할 때는 보통 코드를 고치기보다 config와 실행 옵션을 먼저 바꿉니다.

가장 자주 바꾸는 것은 아래 항목입니다.

| 하고 싶은 일 | 바꿀 곳 | 예시 |
|---|---|---|
| 다른 실험 이름으로 저장 | config의 `experiment.name` 또는 `artifact_policy.run_id` | `rag_hybrid_v2`, `run_001` |
| 다른 데이터 사용 | config의 `paths.data_dir` 또는 `paths.raw_docs_dir` | `data/my_dataset`, `data/raw_docs` |
| 다른 모델 사용 | config의 `model.name`, `model.model_name` | `huggingface_sequence_classifier` |
| RAG 검색 방식 변경 | config의 `rag.retriever.method` | `keyword`, `semantic`, `hybrid` |
| 검색 결과 개수 변경 | config의 `rag.retriever.top_k` | `3`, `5`, `10` |
| chunk 크기 변경 | config의 `rag.chunk.size`, `rag.chunk.overlap` | `500`, `80` |
| 일부 단계만 실행 | 이 노트북의 `RUN_*` 옵션 | `RUN_RAG_COMPARE = False` |

권장 방식은 config 파일을 복사해서 새 이름으로 만든 뒤, 이 노트북의 `TEXT_CONFIG` 또는 `RAG_CONFIG`를 새 config 경로로 바꾸는 것입니다.

예시:

```python
RAG_CONFIG = Path("configs/experiments/rag/rag_smoke_hybrid.yaml")
RUN_RAG_SMOKE = True
RUN_RAG_COMPARE = True
```

주의할 점:

- 같은 `experiment.name`을 계속 쓰면 결과 폴더가 덮어써질 수 있습니다.
- 여러 번 비교하려면 `artifact_policy.run_id`를 사용하면 좋습니다.
- 실제 데이터가 들어오면 먼저 loader/chunking 결과를 확인한 뒤 retriever와 metric을 비교합니다.


## 2. 실행 옵션

아래 값으로 어떤 단계를 실행할지 정합니다.

처음 실행할 때는 기본값 그대로 두면 됩니다. 이미 확인한 단계는 `False`로 바꾸면 시간을 줄일 수 있습니다.

In [ ]:
# True면 실행, False면 건너뜁니다.
# 아래 RUN_* 값으로 실행할 단계를 선택합니다.
RUN_VALIDATE = True
RUN_TEXT_SMOKE = True
# HuggingFace 실행 환경을 확인하고 싶으면 True로 바꿉니다.
RUN_HF_TINY = False
RUN_RAG_SMOKE = True
RUN_RAG_COMPARE = True
RUN_SUMMARY = True

# 이번 노트북에서 사용할 config와 입력 파일입니다.
# 텍스트 분류 실험 config입니다. 새 분류 실험을 만들면 이 경로를 바꿉니다.
TEXT_CONFIG = Path("configs/smoke/smoke_test_text.yaml")
HF_TINY_CONFIG = Path("configs/smoke/smoke_test_hf_tiny.yaml")
# RAG 실험 config입니다. retriever/chunk/top_k 실험을 바꾸려면 새 config를 지정합니다.
RAG_CONFIG = Path("configs/experiments/rag/rag_smoke_test.yaml")
PREDICT_INPUT = Path("data/text_processed/sample_positive.txt")
# RAG 단건 질문 테스트에 사용할 질문입니다.
QUESTION = "예산은 얼마야?"

# 파일이 실제로 있는지 먼저 확인합니다.
for path in [TEXT_CONFIG, HF_TINY_CONFIG, RAG_CONFIG, PREDICT_INPUT]:
    print(path, "exists=", path.exists())

## 3. 공통 함수

아래 함수들은 반복 작업을 쉽게 하려고 만든 도구입니다.

- `run`: 터미널 명령 실행
- `show_json`: JSON 결과 보기
- `show_csv`: CSV 결과를 표로 보기
- `plot_history`: 학습 곡선 그리기
- `plot_metric_bars`: metric 막대그래프 그리기

In [ ]:
def run(command: str, check: bool = True) -> subprocess.CompletedProcess:
    # 터미널 명령을 노트북 안에서 실행하고 결과를 바로 보여주는 함수입니다.
    print("\n$", command)
    result = subprocess.run(command, shell=True, text=True, capture_output=True)

    # 표준 출력과 에러를 모두 노트북에 남겨야 나중에 실패 원인을 찾기 쉽습니다.
    if result.stdout:
        print(result.stdout)
    if result.stderr:
        print(result.stderr)

    # 명령이 실패했는데도 다음 셀이 계속 실행되면 원인이 더 헷갈릴 수 있어서 바로 멈춥니다.
    if check and result.returncode != 0:
        raise RuntimeError(f"Command failed with exit code {result.returncode}: {command}")
    return result


def read_json(path: str | Path) -> dict:
    # metrics.json, run_status.json 같은 결과 파일을 dict로 읽습니다.
    path = Path(path)
    if not path.exists():
        print("missing:", path)
        return {}
    return json.loads(path.read_text(encoding="utf-8"))


def show_json(path: str | Path) -> dict:
    # JSON 결과를 사람이 읽기 좋게 출력합니다.
    payload = read_json(path)
    print(json.dumps(payload, ensure_ascii=False, indent=2))
    return payload


def show_csv(path: str | Path, n: int = 20) -> pd.DataFrame:
    # CSV 결과를 표 형태로 확인합니다.
    path = Path(path)
    if not path.exists():
        print("missing:", path)
        return pd.DataFrame()
    frame = pd.read_csv(path)
    display(frame.head(n))
    return frame


def plot_history(experiment_dir: str | Path) -> pd.DataFrame:
    # history.csv에는 epoch별 metric이 들어갑니다.
    # 이 그래프를 보면 학습이 좋아지는지, 멈춰 있는지, 흔들리는지 볼 수 있습니다.
    history_path = Path(experiment_dir) / "history.csv"
    history = show_csv(history_path)
    if history.empty:
        return history

    metric_cols = [
        col for col in history.columns
        if col != "epoch" and pd.api.types.is_numeric_dtype(history[col])
    ]
    if metric_cols:
        ax = history.plot(x="epoch", y=metric_cols, marker="o", figsize=(8, 4))
        ax.set_title(f"Learning Curve: {Path(experiment_dir).name}")
        ax.set_xlabel("epoch")
        ax.set_ylabel("metric")
        plt.show()
    return history


def plot_metric_bars(metrics: dict, title: str) -> None:
    # metrics.json의 숫자 지표를 막대그래프로 봅니다.
    # 1에 가까울수록 좋은 지표가 많지만, 지표마다 의미는 config와 문서를 함께 확인해야 합니다.
    numeric = {key: value for key, value in metrics.items() if isinstance(value, (int, float))}
    if not numeric:
        print("numeric metric이 없습니다.")
        return

    ax = pd.Series(numeric).sort_index().plot(kind="bar", figsize=(8, 4), ylim=(0, 1.05))
    ax.set_title(title)
    ax.set_ylabel("score")
    plt.xticks(rotation=30, ha="right")
    plt.show()


## 4. 환경 확인

Torch와 GPU 상태를 확인합니다.

CPU만 보여도 smoke test는 돌 수 있습니다. 다만 HuggingFace fine-tuning은 GPU가 있어야 훨씬 빠릅니다.

In [ ]:
try:
    import torch

    print("Torch:", torch.__version__)
    print("CUDA:", torch.cuda.is_available())
    print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")
except ImportError:
    print("torch가 설치되어 있지 않습니다. HuggingFace 실험 전 requirements를 확인하세요.")

## 5. Config 미리보기

실험을 돌리기 전에 설정을 먼저 봅니다.

여기서 실험 이름, 출력 폴더, 모델, RAG 검색 방식이 기대와 다르면 config 파일을 먼저 수정해야 합니다.

In [ ]:
import yaml

for config_path in [TEXT_CONFIG, RAG_CONFIG]:
    print("\n==", config_path, "==")
    config = yaml.safe_load(config_path.read_text(encoding="utf-8"))

    # 전체 config는 길 수 있으니 중요한 부분만 골라서 봅니다.
    print(json.dumps({
        "experiment": config.get("experiment"),
        "paths": config.get("paths"),
        "model": config.get("model"),
        "rag_retriever": config.get("rag", {}).get("retriever"),
        "metric": config.get("metric"),
    }, ensure_ascii=False, indent=2))

## 6. 데이터 검증

학습 전에 데이터 파일 구조가 맞는지 확인합니다.

이 단계가 실패하면 모델 문제가 아니라 데이터 준비 문제일 가능성이 큽니다. `train.csv`, `valid.csv`, `test.csv`, `class_map.json`을 먼저 확인하세요.

In [ ]:
if RUN_VALIDATE:
    run("python scripts/run_validate.py --data-dir data/text_processed --project-root .")
else:
    print("skip")

## 7. 텍스트 분류 Smoke Test

가벼운 텍스트 모델로 학습과 예측 흐름이 끝까지 도는지 확인합니다.

여기서 중요한 것은 높은 성능이 아니라, 데이터 로드부터 산출물 저장까지 전체 흐름이 깨지지 않는지입니다.

In [ ]:
if RUN_TEXT_SMOKE:
    # 학습을 실행하고 experiments/smoke_test_text에 결과를 저장합니다.
    run(f"python scripts/run_train.py --config {TEXT_CONFIG} --project-root .")

    # 저장된 모델로 샘플 입력을 예측해봅니다.
    run(f"python scripts/run_predict.py --config {TEXT_CONFIG} --project-root . --input {PREDICT_INPUT}")
else:
    print("skip")

### 텍스트 분류 Metric과 학습 곡선

`metrics.json`은 최종 점수, `history.csv`는 epoch별 변화를 보여줍니다.

실제 실험에서는 학습 곡선이 계속 좋아지는지, 중간에 멈추는지, 갑자기 나빠지는지를 봅니다.

In [ ]:
text_dir = Path("experiments/smoke_test_text")

# 최종 metric을 숫자와 그래프로 확인합니다.
text_metrics = show_json(text_dir / "metrics.json")
plot_metric_bars(text_metrics, "Text Classification Metrics")

# epoch별 metric 변화를 확인합니다.
plot_history(text_dir)

## 8. HuggingFace Tiny Smoke Test

HuggingFace 모델을 실제로 내려받고 학습/저장/예측까지 되는지 확인하는 선택 단계입니다.

처음에는 시간이 걸릴 수 있으니 기본값은 꺼두었습니다. GPU나 transformers 환경을 확인하고 싶을 때 `RUN_HF_TINY = True`로 바꿉니다.

In [ ]:
if RUN_HF_TINY:
    run(f"python scripts/run_train.py --config {HF_TINY_CONFIG} --project-root .")
    run(f"python scripts/run_predict.py --config {HF_TINY_CONFIG} --project-root . --input {PREDICT_INPUT}")

    hf_dir = Path("experiments/smoke_test_hf_tiny")
    plot_metric_bars(show_json(hf_dir / "metrics.json"), "HF Tiny Metrics")
    plot_history(hf_dir)
else:
    print("skip")

## 9. RAG Smoke Test

RAG 파이프라인이 문서를 읽고, 나누고, 검색하고, 답변하고, 평가하는지 확인합니다.

RAG에서는 답변 자체도 중요하지만, 답변의 근거인 citation이 맞는지도 같이 봐야 합니다.

In [ ]:
if RUN_RAG_SMOKE:
    # 문서를 읽고 chunk와 embedding을 만듭니다.
    run(f"python scripts/run_rag_ingest.py --config {RAG_CONFIG} --project-root .")

    # 질문 하나를 넣어 답변과 citation이 나오는지 봅니다.
    run(f"python scripts/run_rag_chat.py --config {RAG_CONFIG} --project-root . --question '{QUESTION}'")

    # 평가 질문 세트로 retrieval/answer/citation 지표를 계산합니다.
    run(f"python scripts/run_rag_chat.py --config {RAG_CONFIG} --project-root . --evaluate")
else:
    print("skip")

### RAG Metric과 평가 결과

RAG 평가는 보통 세 가지를 봅니다.

- 관련 문서를 찾았는가
- 답변에 기대한 내용이 들어갔는가
- citation이 올바른 근거를 가리키는가

In [ ]:
rag_dir = Path("experiments/rag_smoke_test")

rag_metrics = show_json(rag_dir / "metrics.json")
plot_metric_bars(rag_metrics, "RAG Evaluation Metrics")

# 질문별 평가 결과를 표로 확인합니다.
show_csv(rag_dir / "evaluation_results.csv")

# RAG ingest가 어디까지 완료되었는지 확인합니다.
show_json(rag_dir / "rag_ingest_checkpoint.json")

### RAG 답변과 Citation 확인

metric이 좋아도 실제 답변이 어색할 수 있습니다.

이 셀에서는 최근 답변과 근거 정보를 표로 확인합니다. citation이 엉뚱한 문서를 가리키면 loader/chunking/retriever를 의심해야 합니다.

In [ ]:
answers_path = Path("experiments/rag_smoke_test/answers.jsonl")
if answers_path.exists():
    answers = [json.loads(line) for line in answers_path.read_text(encoding="utf-8").splitlines() if line.strip()]
    answers_df = pd.DataFrame(answers)
    display(answers_df.tail(5))
else:
    print("answers.jsonl 없음")

## 10. Retriever 비교

검색 방식을 비교합니다.

keyword, semantic, hybrid 중 어떤 방식이 더 안정적인지 metric으로 봅니다. 실제 데이터에서는 이 결과를 보고 기본 retriever를 정하게 됩니다.

In [ ]:
if RUN_RAG_COMPARE:
    run("python scripts/compare_rag_retrievers.py --project-root .")
else:
    print("skip")

comparison = show_csv("reports/rag_retriever_comparison.csv")
if not comparison.empty:
    metric_cols = ["retrieval_hit_rate", "answer_contains_expected_rate", "citation_correct_rate", "not_found_rate"]
    existing = [col for col in metric_cols if col in comparison.columns]
    comparison.set_index("experiment")[existing].plot(kind="bar", figsize=(9, 4), ylim=(0, 1.05))
    plt.title("RAG Retriever Comparison")
    plt.xticks(rotation=30, ha="right")
    plt.show()

## 11. 실험 요약

여러 실험 결과를 한 표로 모읍니다.

팀 회의나 발표 자료를 만들 때는 이 요약 파일을 기준으로 어떤 실험이 더 나았는지 비교하면 됩니다.

In [ ]:
if RUN_SUMMARY:
    run("python scripts/summarize_experiments.py --project-root .")
else:
    print("skip")

summary = show_csv("reports/experiment_summary.csv")
if not summary.empty:
    display(summary.sort_values("status"))

## 12. 실험 메모

실험을 돌린 뒤에는 아래를 직접 적습니다.

- 이번 실험에서 확인한 점:
- metric에서 볼 점:
- 학습 곡선에서 이상한 점:
- RAG 검색/답변에서 보완할 점:
- 다음 실험에서 바꿀 config: